In [0]:
%pip install "pymongo[srv]" certifi

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./00_setup_mongodb

In [0]:
client = conectar_mongodb()

print("Conectado ao MongoDB Atlas!")

In [0]:
mongo_database = "databricks_learning"

collection_origem = "chamados_insights"
collection_destino = "chamados_dashboard"

db = client[mongo_database]

chamados_insights = db[collection_origem]
chamados_dashboard = db[collection_destino]

print("Database:", mongo_database)
print("Collection origem:", collection_origem)
print("Collection destino:", collection_destino)

In [0]:
total_insights = chamados_insights.count_documents({})

print("Total de documentos em chamados_insights:", total_insights)

if total_insights == 0:
    raise ValueError("Nenhum documento encontrado em chamados_insights.")

In [0]:
def agregacao_para_dict(collection, campo):
    pipeline = [
        {
            "$group": {
                "_id": f"${campo}",
                "quantidade": {"$sum": 1}
            }
        },
        {
            "$sort": {
                "quantidade": -1
            }
        }
    ]

    resultado = {}

    for item in collection.aggregate(pipeline):
        chave = item["_id"] if item["_id"] is not None else "Não informado"
        resultado[str(chave)] = item["quantidade"]

    return resultado

In [0]:
por_risco_sla = agregacao_para_dict(chamados_insights, "risco_sla")
por_status = agregacao_para_dict(chamados_insights, "status")
por_prioridade = agregacao_para_dict(chamados_insights, "prioridade")
por_categoria = agregacao_para_dict(chamados_insights, "categoria")
por_segmento = agregacao_para_dict(chamados_insights, "cliente.segmento")

print("Por risco SLA:", por_risco_sla)
print("Por status:", por_status)
print("Por prioridade:", por_prioridade)
print("Por categoria:", por_categoria)
print("Por segmento:", por_segmento)

In [0]:
pipeline_clientes_criticos = [
    {
        "$match": {
            "risco_sla": "Alto"
        }
    },
    {
        "$group": {
            "_id": "$cliente.nome",
            "quantidade_chamados_criticos": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "quantidade_chamados_criticos": -1
        }
    },
    {
        "$limit": 10
    }
]

top_clientes_criticos = [
    {
        "cliente": item["_id"],
        "quantidade_chamados_criticos": item["quantidade_chamados_criticos"]
    }
    for item in chamados_insights.aggregate(pipeline_clientes_criticos)
]

print(top_clientes_criticos)

In [0]:
pipeline_categorias_criticas = [
    {
        "$match": {
            "risco_sla": "Alto"
        }
    },
    {
        "$group": {
            "_id": "$categoria",
            "quantidade_chamados_criticos": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "quantidade_chamados_criticos": -1
        }
    },
    {
        "$limit": 10
    }
]

top_categorias_criticas = [
    {
        "categoria": item["_id"],
        "quantidade_chamados_criticos": item["quantidade_chamados_criticos"]
    }
    for item in chamados_insights.aggregate(pipeline_categorias_criticas)
]

print(top_categorias_criticas)

In [0]:
from datetime import datetime
import json

total_risco_alto = por_risco_sla.get("Alto", 0)
total_fora_sla = por_risco_sla.get("Encerrado fora do SLA", 0)
total_dentro_sla = por_risco_sla.get("Encerrado dentro do SLA", 0)
total_baixo = por_risco_sla.get("Baixo", 0)
total_medio = por_risco_sla.get("Médio", 0)

percentual_risco_alto = round((total_risco_alto / total_insights) * 100, 2)
percentual_fora_sla = round((total_fora_sla / total_insights) * 100, 2)
percentual_dentro_sla = round((total_dentro_sla / total_insights) * 100, 2)

documento_dashboard = {
    "_id": "snapshot_atual",
    "tipo": "dashboard_chamados",
    "total_chamados": total_insights,
    "indicadores": {
        "total_risco_alto": total_risco_alto,
        "total_risco_medio": total_medio,
        "total_risco_baixo": total_baixo,
        "total_fora_sla": total_fora_sla,
        "total_dentro_sla": total_dentro_sla,
        "percentual_risco_alto": percentual_risco_alto,
        "percentual_fora_sla": percentual_fora_sla,
        "percentual_dentro_sla": percentual_dentro_sla
    },
    "distribuicoes": {
        "por_risco_sla": por_risco_sla,
        "por_status": por_status,
        "por_prioridade": por_prioridade,
        "por_categoria": por_categoria,
        "por_segmento": por_segmento
    },
    "rankings": {
        "top_clientes_criticos": top_clientes_criticos,
        "top_categorias_criticas": top_categorias_criticas
    },
    "origem": {
        "database": mongo_database,
        "collection_origem": collection_origem,
        "processado_por": "databricks",
        "data_processamento": datetime.now()
    }
}

print(json.dumps(documento_dashboard, indent=2, ensure_ascii=False, default=str))

In [0]:
resultado = chamados_dashboard.replace_one(
    {"_id": "snapshot_atual"},
    documento_dashboard,
    upsert=True
)

print("Dashboard snapshot_atual salvo com sucesso!")
print("Matched:", resultado.matched_count)
print("Modified:", resultado.modified_count)
print("Upserted ID:", resultado.upserted_id)

In [0]:
from copy import deepcopy

documento_historico = deepcopy(documento_dashboard)

timestamp_snapshot = datetime.now().strftime("%Y%m%d_%H%M%S")

documento_historico["_id"] = f"snapshot_{timestamp_snapshot}"
documento_historico["tipo"] = "dashboard_chamados_historico"

resultado_historico = chamados_dashboard.insert_one(documento_historico)

print("Snapshot histórico salvo com sucesso!")
print("ID:", resultado_historico.inserted_id)

In [0]:
total_dashboard = chamados_dashboard.count_documents({})

print("Total de documentos em chamados_dashboard:", total_dashboard)

snapshot_atual = chamados_dashboard.find_one({"_id": "snapshot_atual"})

print("Snapshot atual:")
print(json.dumps(snapshot_atual, indent=2, ensure_ascii=False, default=str))